## Pipeline

In [1]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen3-0.6B")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cuda:0


In [2]:
messages = [
    {"role": "user", "content": "/no_think Who are you?"},
]
pipe(messages)

[{'generated_text': [{'role': 'user', 'content': '/no_think Who are you?'},
   {'role': 'assistant',
    'content': "<think>\n\n</think>\n\nI'm a language model. I can assist with various tasks, such as answering"}]}]

In [4]:
messages = [
    {"role": "user", "content": "/no_think Who is president of the USA?"},
]
pipe(messages)

[{'generated_text': [{'role': 'user',
    'content': '/no_think Who is president of the USA?'},
   {'role': 'assistant',
    'content': '<think>\n\n</think>\n\nThe current President of the United States is **Donald Trump**.'}]}]

## Xgrammar json

In [1]:
import xgrammar as xgr

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

device = "cuda"  # Or "cpu", etc.
model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32, device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Introduce yourself in JSON briefly."},
]
texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer(texts, return_tensors="pt").to(model.device)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [2]:
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
grammar_compiler = xgr.GrammarCompiler(tokenizer_info)
compiled_grammar = grammar_compiler.compile_builtin_json_grammar()
# Other ways: provide a json schema string
# compiled_grammar = grammar_compiler.compile_json_schema(json_schema_string)
# Or provide an EBNF string
# compiled_grammar = grammar_compiler.compile_grammar(ebnf_string)

In [3]:
xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
generated_ids = model.generate(
    **model_inputs, max_new_tokens=512, logits_processor=[xgr_logits_processor]
)
generated_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

{"name": "A helpful assistant", "age": 25, "location": "New York", "interests": ["technology", "travel", "culture"], "skills": ["programming", "writing", "design"]}


## Xgrammar ebnf

In [9]:
import xgrammar as xgr

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

device = "cuda"  # Or "cpu", etc.
model_name = "Qwen/Qwen3-0.6B"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32, device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Introduce yourself in JSON briefly."},
]
texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
model_inputs = tokenizer(texts, return_tensors="pt").to(model.device)

In [10]:
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
grammar_compiler = xgr.GrammarCompiler(tokenizer_info)
# Grammar string that represents a JSON schema
json_grammar_ebnf_str = r"""
root ::= basic_array | basic_object
basic_any ::= basic_number | basic_string | basic_boolean | basic_null | basic_array | basic_object
basic_integer ::= ("0" | "-"? [1-9] [0-9]*) ".0"?
basic_number ::= ("0" | "-"? [1-9] [0-9]*) ("." [0-9]+)? ([eE] [+-]? [0-9]+)?
basic_string ::= (([\"] basic_string_1 [\"]))
basic_string_1 ::= "" | [^"\\\x00-\x1F] basic_string_1 | "\\" escape basic_string_1
escape ::= ["\\/bfnrt] | "u" [A-Fa-f0-9] [A-Fa-f0-9] [A-Fa-f0-9] [A-Fa-f0-9]
basic_boolean ::= "true" | "false"
basic_null ::= "null"
basic_array ::= "[" ("" | ws basic_any (ws "," ws basic_any)*) ws "]"
basic_object ::= "{" ("" | ws basic_string ws ":" ws basic_any ( ws "," ws basic_string ws ":" ws basic_any)*) ws "}"
ws ::= [ \n\t]*
"""
compiled_grammar = grammar_compiler.compile_grammar(json_grammar_ebnf_str)

In [11]:
xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
generated_ids = model.generate(
    **model_inputs, max_new_tokens=512, logits_processor=[xgr_logits_processor]
)
generated_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
print(tokenizer.decode(generated_ids, skip_special_tokens=True))

{"name": "A helpful assistant", "age": "25", "occupation": "Software Engineer", "interests": ["Technology", "Nature", "Music"], "description": "A friendly and enthusiastic software engineer with a passion for exploring the world around me."}


## zero shot

In [1]:
import os
import pypdfium2 as pdfium

base_dir = "/pvc/Geschaeftsberichte"
pdf_texts = {}

for root, dirs, files in os.walk(base_dir):
    for f in files:
        if f.lower().endswith(".pdf"):
            pdf_path = os.path.join(root, f)
            pdf = pdfium.PdfDocument(pdf_path)
            pages_text = []
            for i in range(len(pdf)):
                page = pdf[i]
                text = page.get_textpage().get_text_range()
                pages_text.append(text)
            pdf_texts[pdf_path] = pages_text

# pdf_texts is a dict: {filename: [page1_text, page2_text, ...]}

/usr/local/lib/python3.12/dist-packages/pypdfium2/_helpers/textpage.py:80: UserWarning: get_text_range() call with default params will be implicitly redirected to get_text_bounded()
  warnings.warn("get_text_range() call with default params will be implicitly redirected to get_text_bounded()")


In [ ]:
import json

with open("pdf_texts.json", "w", encoding="utf-8") as f:
    json.dump(pdf_texts, f, ensure_ascii=False, indent=2)

In [25]:
import json

with open("pdf_texts.json", "r", encoding="utf-8") as f:
    pdf_texts = json.load(f)

In [2]:
source = pdf_texts['/pvc/Geschaeftsberichte/IBB/ibb_geschaeftsbericht_2012.pdf']
len(source)

152

In [3]:
import os
from huggingface_hub import login

login(token=os.environ["HUGGING_FACE_HUB_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
hgb_guv = '''
Handelsgesetzbuch
§ 275 Gliederung
(1) Die Gewinn- und Verlustrechnung ist in Staffelform nach dem Gesamtkostenverfahren oder dem Umsatzkostenverfahren aufzustellen. Dabei sind die in Absatz 2 oder 3 bezeichneten Posten in der angegebenen Reihenfolge gesondert auszuweisen.
(2) Bei Anwendung des Gesamtkostenverfahrens sind auszuweisen:

1.
    Umsatzerlöse
2.
    Erhöhung oder Verminderung des Bestands an fertigen und unfertigen Erzeugnissen
3.
    andere aktivierte Eigenleistungen
4.
    sonstige betriebliche Erträge
5.
    Materialaufwand:

    a)
        Aufwendungen für Roh-, Hilfs- und Betriebsstoffe und für bezogene Waren
    b)
        Aufwendungen für bezogene Leistungen

6.
    Personalaufwand:

    a)
        Löhne und Gehälter
    b)
        soziale Abgaben und Aufwendungen für Altersversorgung und für Unterstützung,
        davon für Altersversorgung

7.
    Abschreibungen:

    a)
        auf immaterielle Vermögensgegenstände des Anlagevermögens und Sachanlagen
    b)
        auf Vermögensgegenstände des Umlaufvermögens, soweit diese die in der Kapitalgesellschaft üblichen Abschreibungen überschreiten

8.
    sonstige betriebliche Aufwendungen
9.
    Erträge aus Beteiligungen,
    davon aus verbundenen Unternehmen
10.
    Erträge aus anderen Wertpapieren und Ausleihungen des Finanzanlagevermögens,
    davon aus verbundenen Unternehmen
11.
    sonstige Zinsen und ähnliche Erträge,
    davon aus verbundenen Unternehmen
12.
    Abschreibungen auf Finanzanlagen und auf Wertpapiere des Umlaufvermögens
13.
    Zinsen und ähnliche Aufwendungen,
    davon an verbundene Unternehmen
14.
    Steuern vom Einkommen und vom Ertrag
15.
    Ergebnis nach Steuern
16.
    sonstige Steuern
17.
    Jahresüberschuss/Jahresfehlbetrag.

(3) Bei Anwendung des Umsatzkostenverfahrens sind auszuweisen:

1.
    Umsatzerlöse
2.
    Herstellungskosten der zur Erzielung der Umsatzerlöse erbrachten Leistungen
3.
    Bruttoergebnis vom Umsatz
4.
    Vertriebskosten
5.
    allgemeine Verwaltungskosten
6.
    sonstige betriebliche Erträge
7.
    sonstige betriebliche Aufwendungen
8.
    Erträge aus Beteiligungen,
    davon aus verbundenen Unternehmen
9.
    Erträge aus anderen Wertpapieren und Ausleihungen des Finanzanlagevermögens,
    davon aus verbundenen Unternehmen
10.
    sonstige Zinsen und ähnliche Erträge,
    davon aus verbundenen Unternehmen
11.
    Abschreibungen auf Finanzanlagen und auf Wertpapiere des Umlaufvermögens
12.
    Zinsen und ähnliche Aufwendungen,
    davon an verbundene Unternehmen
13.
    Steuern vom Einkommen und vom Ertrag
14.
    Ergebnis nach Steuern
15.
    sonstige Steuern
16.
    Jahresüberschuss/Jahresfehlbetrag.

(4) Veränderungen der Kapital- und Gewinnrücklagen dürfen in der Gewinn- und Verlustrechnung erst nach dem Posten "Jahresüberschuß/Jahresfehlbetrag" ausgewiesen werden.
(5) Kleinstkapitalgesellschaften (§ 267a) können anstelle der Staffelungen nach den Absätzen 2 und 3 die Gewinn- und Verlustrechnung wie folgt darstellen:

1.
    Umsatzerlöse,
2.
    sonstige Erträge,
3.
    Materialaufwand,
4.
    Personalaufwand,
5.
    Abschreibungen,
6.
    sonstige Aufwendungen,
7.
    Steuern,
8.
    Jahresüberschuss/Jahresfehlbetrag.
'''

hgb_bilanz = '''
Handelsgesetzbuch
§ 266 Gliederung der Bilanz
(1) Die Bilanz ist in Kontoform aufzustellen. Dabei haben mittelgroße und große Kapitalgesellschaften (§ 267 Absatz 2 und 3) auf der Aktivseite die in Absatz 2 und auf der Passivseite die in Absatz 3 bezeichneten Posten gesondert und in der vorgeschriebenen Reihenfolge auszuweisen. Kleine Kapitalgesellschaften (§ 267 Abs. 1) brauchen nur eine verkürzte Bilanz aufzustellen, in die nur die in den Absätzen 2 und 3 mit Buchstaben und römischen Zahlen bezeichneten Posten gesondert und in der vorgeschriebenen Reihenfolge aufgenommen werden. Kleinstkapitalgesellschaften (§ 267a) brauchen nur eine verkürzte Bilanz aufzustellen, in die nur die in den Absätzen 2 und 3 mit Buchstaben bezeichneten Posten gesondert und in der vorgeschriebenen Reihenfolge aufgenommen werden.
(2) Aktivseite

A.
    Anlagevermögen:

    I.
        Immaterielle Vermögensgegenstände:

        1.
            Selbst geschaffene gewerbliche Schutzrechte und ähnliche Rechte und Werte;
        2.
            entgeltlich erworbene Konzessionen, gewerbliche Schutzrechte und ähnliche Rechte und Werte sowie Lizenzen an solchen Rechten und Werten;
        3.
            Geschäfts- oder Firmenwert;
        4.
            geleistete Anzahlungen;

    II.
        Sachanlagen:

        1.
            Grundstücke, grundstücksgleiche Rechte und Bauten einschließlich der Bauten auf fremden Grundstücken;
        2.
            technische Anlagen und Maschinen;
        3.
            andere Anlagen, Betriebs- und Geschäftsausstattung;
        4.
            geleistete Anzahlungen und Anlagen im Bau;

    III.
        Finanzanlagen:

        1.
            Anteile an verbundenen Unternehmen;
        2.
            Ausleihungen an verbundene Unternehmen;
        3.
            Beteiligungen;
        4.
            Ausleihungen an Unternehmen, mit denen ein Beteiligungsverhältnis besteht;
        5.
            Wertpapiere des Anlagevermögens;
        6.
            sonstige Ausleihungen.

B.
    Umlaufvermögen:

    I.
        Vorräte:

        1.
            Roh-, Hilfs- und Betriebsstoffe;
        2.
            unfertige Erzeugnisse, unfertige Leistungen;
        3.
            fertige Erzeugnisse und Waren;
        4.
            geleistete Anzahlungen;

    II.
        Forderungen und sonstige Vermögensgegenstände:

        1.
            Forderungen aus Lieferungen und Leistungen;
        2.
            Forderungen gegen verbundene Unternehmen;
        3.
            Forderungen gegen Unternehmen, mit denen ein Beteiligungsverhältnis besteht;
        4.
            sonstige Vermögensgegenstände;

    III.
        Wertpapiere:

        1.
            Anteile an verbundenen Unternehmen;
        2.
            sonstige Wertpapiere;

    IV.
        Kassenbestand, Bundesbankguthaben, Guthaben bei Kreditinstituten und Schecks.

C.
    Rechnungsabgrenzungsposten.
D.
    Aktive latente Steuern.
E.
    Aktiver Unterschiedsbetrag aus der Vermögensverrechnung.

(3) Passivseite

A.
    Eigenkapital:

    I.
        Gezeichnetes Kapital;
    II.
        Kapitalrücklage;
    III.
        Gewinnrücklagen:

        1.
            gesetzliche Rücklage;
        2.
            Rücklage für Anteile an einem herrschenden oder mehrheitlich beteiligten Unternehmen;
        3.
            satzungsmäßige Rücklagen;
        4.
            andere Gewinnrücklagen;

    IV.
        Gewinnvortrag/Verlustvortrag;
    V.
        Jahresüberschuß/Jahresfehlbetrag.

B.
    Rückstellungen:

    1.
        Rückstellungen für Pensionen und ähnliche Verpflichtungen;
    2.
        Steuerrückstellungen;
    3.
        sonstige Rückstellungen.

C.
    Verbindlichkeiten:

    1.
        Anleihen,
        davon konvertibel;
    2.
        Verbindlichkeiten gegenüber Kreditinstituten;
    3.
        erhaltene Anzahlungen auf Bestellungen;
    4.
        Verbindlichkeiten aus Lieferungen und Leistungen;
    5.
        Verbindlichkeiten aus der Annahme gezogener Wechsel und der Ausstellung eigener Wechsel;
    6.
        Verbindlichkeiten gegenüber verbundenen Unternehmen;
    7.
        Verbindlichkeiten gegenüber Unternehmen, mit denen ein Beteiligungsverhältnis besteht;
    8.
        sonstige Verbindlichkeiten,
        davon aus Steuern,
        davon im Rahmen der sozialen Sicherheit.

D.
    Rechnungsabgrenzungsposten.
E.
    Passive latente Steuern.
'''

In [5]:
from abc import ABC, abstractmethod
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

class StringClassifier(ABC):
    def __init__(self, model, model_name):
        self.model_name = model_name
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.config = AutoConfig.from_pretrained(model_name)

    @abstractmethod
    def classify(self, text):
        pass

class BinaryStringClassifier(StringClassifier):
    def __init__(self, model, model_name):
        super().__init__(model, model_name)
        self.valid_token_ids = [self.tokenizer("no", add_special_tokens=False)["input_ids"][0],
                   self.tokenizer("yes", add_special_tokens=False)["input_ids"][0]]

    def prefix_allowed_tokens_fn(self, batch_id, input_ids):
        return self.valid_token_ids

    def get_messages(self, page, law_context = False):
        messages = [{"role": "system", "content": "You are a helpful assistant that can classify texts extracted from PDFs."}]

        if law_context:
            messages.append({"role": "system", "content": f"You know the laws about how to structure the 'Gewinn- und Verlustrechnung' (profit and loss statement) table:' \n\n'''\n{hgb_guv}\n'''."})

        messages.append({"role": "user", "content": f"Bewerte, ob der folgende Text die Tabelle zur 'Gewinn- und Verlustrechnung' (profit and loss statement) enthält: \n\n'''\n{page}\n'''."})
        return messages

    def classify(self, text, law_context = False):
        messages = self.get_messages(text, law_context)
        texts = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer(texts, return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=1,
            prefix_allowed_tokens_fn=self.prefix_allowed_tokens_fn,
            pad_token_id=self.tokenizer.eos_token_id
        )

        print(f"page {start+i}: {self.tokenizer.decode(generated_ids[0][-1], skip_special_tokens=True)}")

class FiveClassStringClassifier(StringClassifier):
    def __init__(self, model, model_name):
        super().__init__(model, model_name)
        self.valid_token_ids = [
            self.tokenizer("Aktiva", add_special_tokens=False)["input_ids"][0],
            self.tokenizer("GuV", add_special_tokens=False)["input_ids"][0],
            self.tokenizer("notable", add_special_tokens=False)["input_ids"][0],
            self.tokenizer("othertable", add_special_tokens=False)["input_ids"][0],
            self.tokenizer("Passiva", add_special_tokens=False)["input_ids"][0]
        ]

    def prefix_allowed_tokens_fn(self, batch_id, input_ids):
        return self.valid_token_ids

    def get_messages(self, page, law_context = False):
        messages = [
            {"role": "system", "content": "You are a helpful assistant that can classify texts extracted from PDFs. You can differentiate between those five categories: 'Aktiva', 'GuV', 'notable', 'othertable', and 'Passiva'."},
            {"role": "system", "content": """
            1) Wenn der vorliegende Text eine Tabelle zur 'Gewinn- und Verlustrechnung' (profit and loss statement) enthält, antworte mit 'GuV'.\n\n
            2) Wenn der Text eine zur Bilanz (balance sheet) gehörige Tabelle zu 'Aktiva' (assets) enthält, antworte mit 'Aktiva'.\n\n
            3) Wenn der Text eine zur Bilanz (balance sheet) gehörige Tabelle zu 'Passiva' (liabilities) enthält, antworte mit 'Passiva'.\n\n
            4) Wenn der Text eine andere Tabelle enthält, antworte mit 'othertable'.\n\n
            5) Wenn der Text keine Tabelle enthält, antworte mit 'notable'.
            """}
        ]

        if law_context:
            messages.append({"role": "system", "content": f"You know the laws about how to structure the 'Gewinn- und Verlustrechnung' (profit and loss statement) table:' \n\n'''\n{hgb_guv}\n'''."})
            messages.append({"role": "system", "content": f"You also know the laws about how to structure the 'Aktiva' (assets) and 'Passiva' (liabilities) table for a 'Bilanz' (balance sheet):' \n\n'''\n{hgb_bilanz}\n'''."})

        messages.append({"role": "user", "content": f"Bestimme, was der folgende Text repräsentiert:' \n\n'''\n{page}\n'''."})
        return messages

    def classify(self, text, law_context = False):
        messages = self.get_messages(text, law_context)
        texts = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer(texts, return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=1,
            prefix_allowed_tokens_fn=self.prefix_allowed_tokens_fn,
            pad_token_id=self.tokenizer.eos_token_id
        )

        print(f"page {start+i}: {self.tokenizer.decode(generated_ids[0][-1], skip_special_tokens=True)}")

In [31]:
device = "cuda"  # Or "cpu", etc.
model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float32, device_map=device
)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [7]:
binary_classifier = BinaryStringClassifier(model, model_name)

start = 110
end = 120

for i, page in enumerate(source[start:end]):
    if len(page) > 0:        
        binary_classifier.classify(page)

page 110: no
page 111: no
page 112: no
page 113: no
page 114: no
page 115: no
page 116: no
page 117: yes
page 118: yes
page 119: no


In [8]:
for i, page in enumerate(source[start:end]):
    if len(page) > 0:        
        binary_classifier.classify(page, law_context=True)

page 110: no
page 111: no
page 112: no
page 113: no
page 114: no
page 115: no
page 116: no
page 117: yes
page 118: yes
page 119: no


## one shot (Gesetzestext)

In [ ]:
five_class_classifier = FiveClassStringClassifier(model, model_name)

start = 110
end = 120

for i, page in enumerate(source[start:end]):
    if len(page) > 0:        
        five_class_classifier.classify(page)

page 110: not
page 111: not
page 112: not
page 113: not
page 114: not
page 115: Akt
page 116: Pass
page 117: Gu
page 118: Gu
page 119: not


In [15]:
five_class_classifier = FiveClassStringClassifier(model, model_name)

start = 110
end = 120

for i, page in enumerate(source[start:end]):
    if len(page) > 0:        
        five_class_classifier.classify(page, law_context=True)

page 110: not
page 111: not
page 112: not
page 113: not
page 114: not
page 115: Akt
page 116: Pass
page 117: Gu
page 118: Gu
page 119: Gu


## classifying texts

In [26]:
import pandas as pd

data = pd.read_csv("../benchmark_truth/table_detection.csv")
data["page"] = data["page"].astype(int)
data.drop(columns=['Unnamed: 0'], inplace=True)
# data

df_table_type = pd.read_csv("../benchmark_truth/aktiva_passiva_guv_table_pages.csv")
# df_table_type

In [27]:
pages_classified = {
    "Aktiva": [],
    "GuV": [],
    "Passiva": [],
    "othertable": [],
    "notable": []
}

for i, row in data.iterrows():
    path = row['filepath']
    text = pdf_texts[path.replace('..', '/pvc')][row["page"]-1]
    page = row["page"]

    entry = {
        "filepath": path,
        "page": page,
        "text": text
    }
    
    if row['table'] == 0:
        pages_classified["notable"].append(entry)
    elif row['table'] == 1:
        if row['target'] == 0:
            pages_classified["othertable"].append(entry)
        elif row['target'] == 1:
            table_type = df_table_type.query(f"page == {page} and filepath == '{path}'")['type'].values[0]

            match table_type:
                case "Aktiva":
                    pages_classified["Aktiva"].append(entry)
                case "GuV":
                    pages_classified["GuV"].append(entry)
                case "Passiva":
                    pages_classified["Passiva"].append(entry)
                case "Aktiva&Passiva":
                    pages_classified["Aktiva"].append(entry)
                    pages_classified["Passiva"].append(entry)
                case _:
                    raise ValueError(f"Unknown table type: {table_type}")

In [9]:
for key, value in pages_classified.items():
    print(f'{key}: {len(value)}')

Aktiva: 87
GuV: 91
Passiva: 86
othertable: 207
notable: 240


In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_token_length = max(
    len(tokenizer(text, add_special_tokens=False)["input_ids"])
    for texts in pages_classified.values()
    for text in texts
)
print("Maximum token length:", max_token_length)

Maximum token length: 4311


In [5]:
embedding_model_name = "BAAI/bge-m3"

In [ ]:
from sentence_transformers import SentenceTransformer

# model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-7B-instruct", trust_remote_code=True) # needs h200!
# model = SentenceTransformer("Linq-AI-Research/Linq-Embed-Mistral")
model = SentenceTransformer(embedding_model_name)

In [ ]:
entries = [dict(entry, type=key) for key, entries in pages_classified.items() for entry in entries]

In [30]:
import json

with open("entries.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, ensure_ascii=False, indent=2)

In [1]:
import json

with open("entries.json", "r", encoding="utf-8") as f:
    entries = json.load(f)

In [2]:
page_texts = [entry['text'] for entry in entries]

In [4]:
embeddings = model.encode(page_texts, show_progress_bar=True) # needs 5 minutes (Qwen2-7B) or 1 minute (bge-m3)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day"
]
embeddings_test = model.encode(sentences)

similarities = model.similarity(embeddings_test, embeddings_test)
print(similarities)
# [4, 4]

tensor([[1.0000, 0.7279, 0.9520, 0.4537],
        [0.7279, 1.0000, 0.7071, 0.3929],
        [0.9520, 0.7071, 1.0000, 0.4184],
        [0.4537, 0.3929, 0.4184, 1.0000]])


In [6]:
import numpy as np

np.save("embeddings.npy", embeddings)

In [3]:
import numpy as np

embeddings = np.load("embeddings.npy")

In [6]:
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name= embedding_model_name
)

# Create or load a ChromaDB collection
client = chromadb.PersistentClient(path="./chroma_db")  # Directory to store the database
collection = client.get_or_create_collection("pages", embedding_function=sentence_transformer_ef)

In [7]:

# Prepare data for insertion
# page_texts is the list of texts, embeddings is the numpy array of embeddings
ids = [entry['type']+str(i) for i, entry in enumerate(entries)]
metadatas = [{"filepath": entry['filepath'], "page": entry['page'], "type": entry['type']} for entry in entries]

collection.add(
    embeddings=embeddings.tolist(),
    documents=page_texts,
    metadatas=metadatas,
    ids=ids
)

In [9]:
collection.query(
    query_texts=["doc10", "thus spake zarathustra"],
    n_results=10
)

{'ids': [['Aktiva64',
   'Aktiva65',
   'GuV153',
   'GuV154',
   'GuV155',
   'Passiva241',
   'Passiva242',
   'Aktiva63',
   'Passiva240',
   'Aktiva68'],
  ['Aktiva63',
   'Passiva240',
   'Aktiva64',
   'Aktiva65',
   'GuV153',
   'GuV154',
   'GuV155',
   'Passiva241',
   'Passiva242',
   'othertable410']],
 'embeddings': None,
 'documents': [['', '', '', '', '', '', '', '', '', 'Anlage II'],
  ['',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '',
   '34\r\nERLÄUTERUNGEN ZUR \r\nVERMÖGENS-, FINANZ- UND ERTRAGSLAGE\r\nVERMÖGENSLAGE DES GESOBAU-KONZERNS UND DER GESOBAU AG\r\nIn den folgenden Übersichten zur Vermögenslage sind die einzelnen Vermögens- und \r\nSchuldposten nach wirtschaftlichen und finanziellen Gesichtspunkten zusammengefasst.\r\n2.5\r\n2.5.1\r\nDer Anstieg der Bilanzsumme und des Sachanlagevermögens ist vornehmlich durch die Bautä\x02tigkeit sowie Bestandsankäufe begründet. Das langfristige Vermögen ist im Wesentlichen mit \r\nlangfristigem Kapital finanziert